# Track model metrics with MLfLow

In [ ]:
! env | grep MLFLOW_TRACKING_URI

In [ ]:
import os
import mlflow
from mlflow.models import infer_signature

import pandas as pd
import time
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Enable System Metrics Logging

In [ ]:
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"
os.environ["MLFLOW_SYSTEM_METRICS_SAMPLING_INTERVAL"] = "1"

## Load Dataset

In [ ]:
# Load the Iris dataset
iris = datasets.load_iris(as_frame=True)
X = iris.data.loc[:, ['sepal length (cm)', 'petal length (cm)']]
y = iris.target

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Train Model

In [ ]:
import mlflow
import pandas as pd
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from uuid import uuid4
import os
import time

# Enable autologging
mlflow.sklearn.autolog()

USER = os.environ['JUPYTERHUB_USER']
EXPERIMENT_NAME = f'{USER}-iris-classifier-model'
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=f'iris-grid-search-{uuid4()}'):
    # Define the parameter grid for grid search
    param_grid = {
        "penalty": ["l1", "l2"],
        "C": [0.1, 1, 10, 100],
        "solver": ["liblinear", "saga"],
        "max_iter": [10_000]
    }
    
    # Create a base model
    base_model = LogisticRegression(random_state=8889)
    
    # Perform grid search
    grid_search = GridSearchCV(estimator=base_model, param_grid=param_grid, cv=5, 
                               scoring='accuracy', n_jobs=-1, return_train_score=True)
    
    # Fit the grid search to the data
    print("Starting grid search...")
    start_time = time.time()
    grid_search.fit(X_train, y_train)
    end_time = time.time()
    print(f"Grid search completed in {end_time - start_time:.2f} seconds")
    
    # Get the best model and its parameters
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    print("Best parameters:", best_params)
        
    # Predict on the test set using the best model
    y_pred = best_model.predict(X_test)
        
    # Log all results as a CSV file (this is not covered by autologging)
    results_df = pd.DataFrame(grid_search.cv_results_)
    results_path = "grid_search_results.csv"
    results_df.to_csv(results_path, index=False)
    mlflow.log_artifact(results_path)
    mlflow.log_artifact('07a-mlflow-tracking.ipynb')

**View the results at: https://nebari.openteams.ai/mlflow**

---

Next: [Run a privately hosted LLM →](./08-llm.ipynb)